# Phenotype generation: doses based on peaks

In [ ]:
import pyspark
import dxpy
import hail as hl
import math
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import sys
import json
import seaborn as sns 
from scipy.signal import find_peaks
from matplotlib.backends.backend_pdf import PdfPages
from tqdm.auto import tqdm  

In [ ]:
sc = pyspark.SparkContext()
spark = pyspark.sql.SparkSession(sc)
hl.init(sc=sc, default_reference='GRCh38')

In [ ]:
input_database = 'prescriptions_db'
input_tb = 'doses_values_phenotypes_v6_2_0.ht'

In [ ]:
input_db_id = dxpy.find_one_data_object(name=input_database, classname='database', project=dxpy.PROJECT_CONTEXT_ID)['id']
ht = hl.read_table(f'dnax://{input_db_id}/{input_tb}')

In [ ]:
phenos = [x for x in ht.row if x != 'eid' and not x.endswith('__ln')]

In [ ]:
ht = ht.select(*phenos)
ht = ht.persist()

In [ ]:
pheno_peaks = {}

In [ ]:
for p in tqdm(phenos, desc="Progress:"):
    
    if p in pheno_peaks:
        continue
    
    stats = ht.aggregate(hl.agg.stats(ht[p]))
    
    if stats.n == 0 or stats.min is None or stats.min == stats.max:
        continue

    hist_data = ht.aggregate(hl.agg.hist(ht[p], stats.min, stats.max, 100))
    
    counts = np.array(hist_data.bin_freq)
    
    peaks, _ = find_peaks(counts, prominence=50, distance=5)

    bin_centers = [(hist_data.bin_edges[i] + hist_data.bin_edges[i+1]) / 2 for i in peaks]
    
    pheno_peaks[p] = {
        'peak_indices': peaks,
        'peak_doses': bin_centers,
        'peak_counts': counts[peaks]
    }

In [ ]:
for p, data in pheno_peaks.items():
    doses = sorted(data['peak_doses'])
    
    if not doses:
        continue
        
    intervals = []
    
    stats = ht.aggregate(hl.agg.stats(ht[p]))
    midpoints = [(doses[i] + doses[i+1]) / 2 for i in range(len(doses) - 1)]
    bounds = [stats.min] + midpoints + [stats.max]
    
    for i in range(len(doses)):
        intervals.append({
            'peak': doses[i],
            'range': (bounds[i], bounds[i+1])
        })
    
    pheno_peaks[p]['intervals'] = intervals

In [ ]:
rows = []

for p, data in pheno_peaks.items():
    if 'intervals' not in data:
        continue
        
    for i, interval in enumerate(data['intervals']):
        rows.append({
            'phenotype': p,
            'peak_value': interval['peak'],
            'lower_bound': interval['range'][0],
            'upper_bound': interval['range'][1],
            'n_patients': data['peak_counts'][i] 
        })

df_peaks = pd.DataFrame(rows)

df_peaks.to_csv('../data/phenotypes_peaks_metadata.csv', index=False)

In [ ]:
available_phenos = [p for p in phenos if p in pheno_peaks and 'intervals' in pheno_peaks[p]]

batch_size = 50

for i in range(0, len(available_phenos), batch_size):
    batch = available_phenos[i : i + batch_size]
    new_annotations = {}
    
    for p in batch:
        p_data = pheno_peaks[p]
        case_expr = hl.case()
        
        for row in p_data['intervals']:
            low, high = row['range']
            peak_val = row['peak']
            case_expr = case_expr.when((ht[p] >= low) & (ht[p] < high), peak_val)
        
        case_expr = case_expr.default(hl.missing(hl.tfloat64))
        new_annotations[f"{p}__peak"] = case_expr
    
    ht = ht.annotate(**new_annotations)
    ht = ht.persist()
    
    print(f"Batch {i//batch_size + 1} done: added {len(new_annotations)} columns and persisted.")

In [ ]:
agg_exprs = [hl.agg.count_where(hl.is_defined(ht[p])) for p in phenos]

counts_list = ht.aggregate(agg_exprs)

df_counts = pd.DataFrame({
    'pheno': phenos,
    'count': counts_list
})

top_100_df = df_counts.sort_values('count', ascending=False).head(100)
top_100_phenos = top_100_df['pheno'].tolist()

In [ ]:
cols_to_keep = [c for c in ht.row if c.endswith('__peak')]
ht_final = ht.select(*cols_to_keep)
ht_final = ht_final.persist()

In [ ]:
output_database = 'prescriptions_db'
output_tb = 'doses_peaks_phenotypes_v6_2_0.ht'

In [ ]:
spark.sql(f"CREATE DATABASE IF NOT EXISTS {output_database} LOCATION 'dnax://'")
output_db_id = dxpy.find_one_data_object(name=output_database, classname='database', project=dxpy.PROJECT_CONTEXT_ID)['id']
output_url = f'dnax://{output_db_id}/{output_tb}'

%time ht_final.write(output_url, overwrite=True)